# End-to-End GNN AML Pipeline

This notebook demonstrates the complete unsupervised AML detection pipeline using a Graph Neural Network (GAT).

### Pipeline Stages:
1.  **Data Loading:** Synthetic transaction generation.
2.  **Graph Construction:** Mapping tabular data to a `HeteroData` object.
3.  **Model Training:** Training a `FraudGAT` model using link prediction (self-supervised).
4.  **Anomaly Detection:** Running `IsolationForest` on the learned node embeddings.

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.gnn.graph.builder import GraphBuilder
from src.gnn.config.schema import Schema
from src.gnn.modeling.training import train_model
from src.gnn.analysis.inference import detect_anomalies

print("Libraries and project modules loaded successfully.")

## 1. Synthetic Data Generation

In [ ]:
def generate_mock_data(num_txns=500):
    data = {
        Schema.TRANSACTION_ID: [f'txn_{i}' for i in range(num_txns)],
        Schema.DATE: pd.date_range(start='2023-01-01', periods=num_txns, freq='10min'),
        Schema.VOLUME: np.random.uniform(10, 10000, num_txns),
        Schema.DIRECTION: np.random.choice(['Inbound', 'Outbound'], num_txns),
        Schema.CUSTOMER_NAME: np.random.choice(['Cust_A', 'Cust_B', 'Cust_C', 'Cust_D', 'Mule_Node'], num_txns),
        Schema.BANK_NAME: np.random.choice(['Bank_Alpha', 'Bank_Beta'], num_txns),
        Schema.BANK_ACCOUNT: np.random.choice([f'Acc_{i}' for i in range(20)], num_txns),
        Schema.RARITY_SCORE: np.random.uniform(0, 1, num_txns),
        Schema.TIMESTAMP_NORM: np.linspace(0, 1, num_txns)
    }
    return pd.DataFrame(data)

df = generate_mock_data()
print(f"Generated {len(df)} transactions.")

## 2. Graph Construction
We convert the dataframe into a `HeteroData` object, including reverse edges for bidirectional message passing.

In [ ]:
builder = GraphBuilder()
graph = builder.build(df)
print(graph)

## 3. Model Training (Self-Supervised Link Prediction)
We train the `FraudGAT` model to learn structural representations of customers and accounts.

In [ ]:
print("Starting training...")
model, embeddings = train_model(graph, hidden_channels=32, epochs=30, lr=0.01)
print("Training complete.")

## 4. Anomaly Detection
We run the Isolation Forest on the learned customer embeddings to find outliers.

In [ ]:
results = detect_anomalies(embeddings, builder.idx_to_customer, contamination=0.1)
print("Detection complete.")

# Display top 10 suspicious customers
print("\n--- Top Anomalies (Lower GNN Score = More Suspicious) ---")
results.head(10)

## 5. Result Visualization
Let's plot the distribution of anomaly scores.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(results['gnn_anomaly_score'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of GNN Anomaly Scores')
plt.xlabel('Isolation Forest Score (Lower is more anomalous)')
plt.ylabel('Frequency')
plt.show()